# Image classification on SageMaker

Fine-tune the built-in image classification algorithm on your own images, evaluate it, and deploy
to the endpoint the CDK stack wires the Lambda to.

In [ ]:
%pip install -q sagemaker boto3 Pillow matplotlib

In [ ]:
import sagemaker, boto3, json
session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "REPLACE-WITH-ImagesBucket-OUTPUT"
region = session.boto_region_name
print(region, bucket)

## 1. Inspect the data first

Class balance and image sizes decide most of the hyperparameters. A class with 30 images and one
with 3,000 will give you a classifier that has learned the prior, not the pictures.

In [ ]:
!python ../src/data/build_manifests.py --bucket {bucket} --prefix raw --out-prefix manifests

In [ ]:
import collections
s3 = boto3.client("s3")
index = json.loads(s3.get_object(Bucket=bucket, Key="manifests/class_index.json")["Body"].read())
manifest = s3.get_object(Bucket=bucket, Key="manifests/train.manifest")["Body"].read().decode().splitlines()
counts = collections.Counter(json.loads(line)["class-metadata"]["class-name"] for line in manifest if line.strip())
print(index["classes"]); print(counts)

## 2. Train

Start with ResNet-50 and pretrained weights. If validation accuracy plateaus well below training
accuracy, the answer is usually more augmentation or fewer layers — not more epochs.

In [ ]:
!python ../src/training/launch.py --bucket {bucket} --role-arn {role} --epochs 15 --instance-type ml.g4dn.xlarge --wait

## 3. Evaluate on the validation manifest

Per-class recall matters more than overall accuracy: one weak class is invisible in the average.

In [ ]:
# after deploying, score the validation set and build a confusion matrix
import numpy as np
from collections import defaultdict

runtime = boto3.client("sagemaker-runtime")
endpoint = "image-classifier-dev"
classes = index["classes"]

val = s3.get_object(Bucket=bucket, Key="manifests/validation.manifest")["Body"].read().decode().splitlines()
confusion = np.zeros((len(classes), len(classes)), dtype=int)

for line in val[:200]:
    record = json.loads(line)
    key = record["source-ref"].split(f"{bucket}/")[1]
    payload = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    probs = json.loads(runtime.invoke_endpoint(EndpointName=endpoint, ContentType="application/x-image", Body=payload)["Body"].read())
    confusion[record["class"], int(np.argmax(probs))] += 1

print(confusion)
print("per-class recall:", (confusion.diagonal() / np.clip(confusion.sum(axis=1), 1, None)).round(3))

## 4. Deploy and try the pipeline end to end

In [ ]:
!python ../src/training/launch.py --deploy --job-name <training-job-name> \
    --bucket {bucket} --role-arn {role} --endpoint-name image-classifier-dev --serverless

!aws s3 cp ./sample.jpg s3://{bucket}/incoming/sample.jpg
!aws dynamodb get-item --table-name image-classifier-dev-predictions \
    --key '{{"imageKey": {{"S": "incoming/sample.jpg"}}}}'